# bn-weight-bias-init-pattern — worked example 3: Prove non-BN layers are untouched by the init

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bn-weight-bias-init-pattern`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The DCGAN BN init must mutate ONLY BatchNorm submodules. A correct `init_fn` guarded by `isinstance(..., (nn.BatchNorm1d, nn.BatchNorm2d))` leaves Conv and Linear weights byte-for-byte identical. We can prove this by snapshotting a non-BN weight before and after `model.apply`.

## Worked solution

**Step 1 — snapshot a Conv weight.** Before applying any init, we `.clone()` the first Conv layer's weight. `.clone()` is essential — without it we would hold a reference that mutates alongside the live tensor and the comparison would be meaningless.

**Step 2 — define the guarded init and apply it.** `init_fn` resamples gamma and zeros beta *only* when the module is a BN type. `model.apply(init_fn)` walks the whole tree, hitting Conv, BN, and Linear, but the guard means the Conv weight is never written.

**Step 3 — compare.** `t.equal(conv_before, model[0].weight)` must be `True`: the Conv weight is unchanged. Meanwhile the BN bias is now all zeros, proving the init *did* run on the BN layer. This pinpoints the contract: BN touched, everything else preserved.

**Step 4 — why the guard is the whole point.** If we dropped the `isinstance` check, `nn.init.normal_(m.weight, 1.0, 0.02)` would corrupt the carefully Conv-initialized weights (which DCGAN sets separately via `N(0, 0.02)`). The guard is what keeps the two init schemes from colliding.

In [ ]:
import torch.nn as nn

t.manual_seed(0)

model = nn.Sequential(
    nn.Conv2d(3, 6, 3),
    nn.BatchNorm2d(6),
    nn.Conv2d(6, 6, 3),
)

conv_before = model[0].weight.clone()

def init_fn(m):
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

model.apply(init_fn)

conv_unchanged = t.equal(conv_before, model[0].weight)
bn_bias_zero = t.allclose(model[1].bias, t.zeros_like(model[1].bias))
print("conv weight unchanged:", conv_unchanged)
print("bn bias zeroed:", bn_bias_zero)